# 4. Models

### 4.1 Gathering data

In [44]:
import pickle
import numpy as np 
import pandas as pd 
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from tqdm import tqdm

In [27]:
# Gathering data from the stored file
dataloc = '/Users/rathish/Documents/Projects/Opportunity_Application_Ranker/data'
mdata = pd.read_pickle(dataloc + '/cleaned_data/featurizeddata.pkl')
mdata.head()

,opportunity__w2v,candidate__w2v,opportunity__dbert,candidate__dbert
0,"[-0.259, 0.197, 0.195, 0.23, 0.131, -0.265, -0...","[1.0, 0.0, -0.129, 0.226, 0.294, -0.123, -0.18...","[-0.049184397, 0.03635802, 0.052360367, 0.0881...","[0.10243426, 0.1508794, 0.19317816, 0.15368427..."
1,"[-0.259, 0.197, 0.195, 0.23, 0.131, -0.265, -0...","[1.0, 0.0, -0.129, 0.226, 0.294, -0.123, -0.18...","[-0.049184397, 0.03635802, 0.052360367, 0.0881...","[0.10054513, 0.1489926, 0.1913335, 0.15610743,..."
2,"[-0.259, 0.197, 0.195, 0.23, 0.131, -0.265, -0...","[1.0, 0.0, -0.129, 0.226, 0.294, -0.123, -0.18...","[-0.049184397, 0.03635802, 0.052360367, 0.0881...","[0.10054513, 0.1489926, 0.1913335, 0.15610743,..."
3,"[-0.259, 0.197, 0.195, 0.23, 0.131, -0.265, -0...","[1.0, 0.0, -0.129, 0.226, 0.294, -0.123, -0.18...","[-0.049184397, 0.03635802, 0.052360367, 0.0881...","[0.10054513, 0.1489926, 0.1913335, 0.15610743,..."
4,"[-0.259, 0.197, 0.195, 0.23, 0.131, -0.265, -0...","[1.0, 0.0, -0.129, 0.226, 0.294, -0.123, -0.18...","[-0.049184397, 0.03635802, 0.052360367, 0.0881...","[0.10054513, 0.1489926, 0.1913335, 0.15610743,..."


In [28]:
# Defining list of column names that contains the names of the columns, if they belong to the job or candidate

job_column = ['ExternalBriefDescription','ExternalDescription', 'Title', 
              'JobCategoryName']
uid_column = ['OpportunityId', 'ApplicationId']
can_column = ['IsCandidateInternal','BehaviorCriteria', 'MotivationCriteria','EducationCriteria', 'LicenseAndCertificationCriteria', 'SkillCriteria', 'WorkExperiences', 'Educations', 'LicenseAndCertifications', 'Skills', 'Motivations', 'Behaviors', 'StepName', 'StepGroup','pass_first_step'] # Column - 'Tag' Will be added later, StepId has been removed
sel_column = ['IsRejected']

# Defining list of columns based on the type of contents

str_column = ['ExternalBriefDescription', 'ExternalDescription', 'Title', 'JobCategoryName', 'BehaviorCriteria', 'MotivationCriteria', 'EducationCriteria', 'LicenseAndCertificationCriteria', 'SkillCriteria', 'WorkExperiences', 'Educations', 'LicenseAndCertifications', 'Skills', 'Motivations', 'Behaviors', 'StepId', 'StepName', 'StepGroup']
bool_column = ['IsCandidateInternal', 'pass_first_step']
float_column = ['Tag']

# Defining list of columns based on the models

model_names = ["w2v", "bert", "use"]

In [29]:
# Gathering necessary arrays and applying Standard Scaller

standardscaler = StandardScaler(copy = False)

In [30]:
# Getting top n-similar cosine vectors
def n_pairwise_cosine_similar(matrix_1, matrix_2, n = 3):

    '''
    Returns diagonal values of cosine similarity and list of index containing that top n similar cosine

    Args:
        matrix_1 (numpy array: 2D): Array containing vectors whose similarity 
        is to be checked
        matrix_2 (numpy array: 2D): Array with which the similarity is to be compared to
        
    n (int, default: 3): number of top similar values to be found

    Returns:
    diag_cosiine_similarity (numpy array): Direct cosine similarity between 
    
    '''
    
    matrix_1 = (matrix_1/np.linalg.norm(matrix_1, axis = 1)[:, np.newaxis])
    matrix_2 = (matrix_2/np.linalg.norm(matrix_2, axis = 1)[:, np.newaxis])
    
    similarity_dict = {}
    cosine_similarity = {}

    for index_1, values_1 in tqdm(enumerate(matrix_1), desc = "Processing", total = len(matrix_1)):
        temp = {}
    
        for index_2, values_2 in enumerate(matrix_2):
            temp_value = np.dot(values_1, values_2)

            if index_1 == index_2:
                cosine_similarity[index_1] = temp_value
                
            temp[index_2] = temp_value

        sorteddict = sorted(
            temp.items(), 
            key = lambda x: x[1], 
            reverse = True
        )[:n]
        
        similarity_dict[index_1] = sorteddict
    
    return cosine_similarity, similarity_dict

In [31]:
# Deriving arrays for w2v vectors

opportunity__w2v = np.array([np.array(x) for x in mdata['opportunity__w2v']])
standardscaler.fit_transform(opportunity__w2v)

candidate__w2v = np.array([np.array(x) for x in mdata['candidate__w2v']])
standardscaler.fit_transform(candidate__w2v)

array([[ 0.14731006, -0.14731006, -0.15264771, ...,  0.96099306,
        -0.96099306, -0.24319345],
       [ 0.14731006, -0.14731006, -0.15264771, ...,  0.96099306,
        -0.96099306, -0.24319345],
       [ 0.14731006, -0.14731006, -0.15264771, ...,  0.96099306,
        -0.96099306, -0.24319345],
       ...,
       [ 0.14731006, -0.14731006, -1.30677646, ...,  0.96099306,
        -0.96099306, -0.24319345],
       [ 0.14731006, -0.14731006, -1.30677646, ...,  0.96099306,
        -0.96099306, -2.2469953 ],
       [ 0.14731006, -0.14731006, -1.30677646, ...,  0.96099306,
        -0.96099306, -2.2469953 ]])

In [32]:
# Reducing the dimensionality of the w2v vectors through PCA

no_of_dimensions = 300
pca = PCA(n_components = no_of_dimensions, copy = False)
  
candidate__w2v = pca.fit_transform(candidate__w2v)
opportunity__w2v = pca.fit_transform(opportunity__w2v)

In [33]:
print(candidate__w2v.shape, opportunity__w2v.shape)

(110267, 300) (110267, 300)


In [34]:
cosine_similarity__w2v, similarity_dict__w2v = n_pairwise_cosine_similar(opportunity__w2v, candidate__w2v, n = 3)

Processing: 100%|██████████| 110267/110267 [3:37:27<00:00,  8.45it/s] 


In [36]:
# Deriving arrays for bert vectors

opportunity__bert = np.array([np.array(x) for x in mdata['opportunity__bert']])
standardscaler.fit_transform(opportunity__w2v)

candidate__bert = np.array([np.array(x) for x in mdata['candidate__bert']])
standardscaler.fit_transform(candidate__w2v)

KeyError: 'opportunity__bert'

In [ ]:
# Reducing the dimensionality of the bert vectors through PCA

no_of_dimensions = 300
pca = PCA(n_components = no_of_dimensions, copy = False)
  
pca.fit_transform(candidate__bert)
pca.fit_transform(opportunity__bert)

In [ ]:
cosine_similarity__bert, similarity_dict__bert = n_pairwise_cosine_similar(opportunity__bert, candidate__bert, n= 3)

In [38]:
# Deriving arrays for dbert vectors

opportunity__dbert = np.array([np.array(x) for x in mdata['opportunity__dbert']])
standardscaler.fit_transform(opportunity__w2v)

candidate__dbert = np.array([np.array(x) for x in mdata['candidate__dbert']])
standardscaler.fit_transform(candidate__w2v)

array([[-0.462679  ,  1.22517228, -1.11896951, ...,  0.27738964,
         0.38144443,  0.44659136],
       [-0.41083368,  0.62894487, -1.9915312 , ..., -0.01599065,
         0.6914494 ,  0.53050205],
       [-0.39569568,  0.58475502, -1.94721415, ..., -0.52724338,
         1.13271871,  0.05982433],
       ...,
       [-0.42844463,  1.15567399,  0.34403837, ...,  0.0986266 ,
        -0.63395705, -0.19463001],
       [-0.51781083,  1.03602418, -0.42861575, ...,  0.35910239,
        -0.18601572,  1.02792873],
       [-0.43641526,  0.88726837, -0.27848351, ...,  0.05638245,
         0.61860625, -0.19203661]])

In [41]:
# Reducing the dimensionality of the dbert vectors through PCA

no_of_dimensions = 300
pca = PCA(n_components = no_of_dimensions, copy = False)
  
candidate__dbert = pca.fit_transform(candidate__dbert)
opportunity__dbert = pca.fit_transform(opportunity__dbert)

In [42]:
cosine_similarity__dbert, similarity_dict__dbert = n_pairwise_cosine_similar(
    opportunity__dbert, candidate__dbert, n= 3
)

Processing: 100%|██████████| 110267/110267 [3:10:44<00:00,  9.63it/s] 


In [45]:
######## Temp Code - To be deleted

# Adding dictionaries into the variables for pickle

# Opening variable.pkl
with open(dataloc + '/cleaned_data/variables.pkl', 'rb') as file:
    variables = pickle.load(file)

# Adding dictionaries
variables["cosine_similarity__w2v"] = cosine_similarity__w2v
variables["similarity_dict__w2v "] = similarity_dict__w2v
variables["cosine_similarity__dbert"] = cosine_similarity__dbert
variables["similarity_dict__dbert"] = similarity_dict__dbert
# variables["cosine_similarity__bert"] = cosine_similarity__bert # Add this line later
# variables["similarity_dict__dbert"] = similarity_dict__dbert # Add this line later

# Saving variables dictionary
with open(dataloc + '/cleaned_data/variables.pkl', 'wb') as file:
    pickle.dump(variables, file)

In [1]:
with open(dataloc + '/cleaned_data/variables.pkl', 'rb') as file:
    variables = pickle.load(file)

print(variables)

NameError: name 'dataloc' is not defined